# Talking avatar on Colab (InfiniteTalk)

Turns a photo + a short voice sample into a talking-head video of that person saying your text,
using this repo's `--engine infinitetalk` with the fast `lightx2v` mode.

**Read this first - what the free tier can and can't do**

- Free Colab gives a **T4 (16 GB)**, ~12 GB of RAM and a few hours of GPU per day, and sometimes no GPU at all.
  InfiniteTalk is a 14B-parameter video model. It was written for much bigger cards, and this notebook has **not been
  tested on a T4**: it may be very slow, run out of memory, or fail.
- So **run cells 1-6 and the one-chunk test (cell 7) before anything else**, and look at how long it takes.
  One chunk is ~3 seconds of video. Multiply by the number of chunks you want.
- Everything you upload (photo, voice sample) goes to Google's servers. Only use your own face and voice, or someone
  who has explicitly agreed.
- Runtime > Change runtime type > **T4 GPU** before you start. Free sessions disconnect and delete `/content`; the
  optional Drive cell keeps finished videos.

## 1. Check the GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU - change the runtime type first")
import shutil
print("free disk: %.0f GB (need ~65 GB)" % (shutil.disk_usage("/content").free / 2**30))

## 2. Get the code

In [ ]:
%%bash
cd /content
if [ ! -d talking-avatar ]; then
  git clone --branch avatar-voice-and-motion-fixes https://github.com/HarukiU06/talking-avatar.git \
    || { echo "Clone failed - if the repo is private, use https://<token>@github.com/... instead"; exit 1; }
fi
cd talking-avatar && git log --oneline -1

## 3. Voice-cloning environment (~5 min)
Same packages as `setup.sh` minus SadTalker, which this notebook doesn't use.

In [ ]:
%%bash
set -e
cd /content/talking-avatar
python3 -m venv .venv
.venv/bin/python -m pip install -q --upgrade pip
.venv/bin/python -m pip install -q torch torchaudio
.venv/bin/python -m pip install -q --no-deps chatterbox-tts
if python3 -c "import sys; sys.exit(0 if sys.version_info >= (3, 13) else 1)"; then NUMPY_SPEC="numpy>=2.0.0"; else NUMPY_SPEC="numpy>=1.24,<2.0"; fi
.venv/bin/python -m pip install -q pyyaml "$NUMPY_SPEC" "librosa==0.11.0" s3tokenizer "transformers==5.2.0" \
  "diffusers==0.29.0" resemble-perth "conformer==0.3.2" "safetensors==0.5.3" spacy-pkuseg "pykakasi==2.3.0" \
  "gradio==6.8.0" pyloudnorm omegaconf noisereduce soundfile
.venv/bin/python -c "import chatterbox, torch; print('chatterbox ok, torch', torch.__version__, torch.cuda.is_available())"

## 4. InfiniteTalk environment (~10 min)
Uses the repo's own setup script, without the parts that don't suit Colab: `flash_attn` (no T4 build; the runner
falls back to PyTorch attention) and its checkpoint downloads (next cell fetches only what the fp8 path reads).

In [ ]:
%%bash
cd /content/talking-avatar
SKIP_FLASH_ATTN=1 SKIP_DOWNLOADS=1 bash setup_infinitetalk.sh 2>&1 | tail -25
ls .venv-infinitetalk/bin/python

## 5. Download the models (~45 GB)
Skips the 7 full-precision diffusion shards and the bf16 T5 - the fp8 path never reads them.

In [ ]:
import subprocess, textwrap
script = textwrap.dedent('''
    from huggingface_hub import snapshot_download
    W = "InfiniteTalk/weights"
    snapshot_download("Wan-AI/Wan2.1-I2V-14B-480P", local_dir=f"{W}/Wan2.1-I2V-14B-480P",
        ignore_patterns=["diffusion_pytorch_model-*", "models_t5_umt5-xxl-enc-bf16.pth", "examples/*"])
    snapshot_download("TencentGameMate/chinese-wav2vec2-base", local_dir=f"{W}/chinese-wav2vec2-base")
    snapshot_download("MeiGen-AI/InfiniteTalk", local_dir=f"{W}/InfiniteTalk",
        allow_patterns=["single/*", "quant_models/infinitetalk_single_fp8*", "quant_models/t5_fp8*",
                        "quant_models/t5_map_fp8*"])
    snapshot_download("Kijai/WanVideo_comfy", local_dir=f"{W}/lora",
        allow_patterns=["Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors"])
    print("downloads done")
''')
r = subprocess.run(["/content/talking-avatar/.venv-infinitetalk/bin/python", "-c", script],
                   cwd="/content/talking-avatar")
print("exit", r.returncode)
import shutil
print("free disk: %.0f GB" % (shutil.disk_usage("/content").free / 2**30))

## 6. Your photo and voice
A clear, front-facing photo works best. The voice sample should be ~10-30 s of clean speech (wav or mp3).

In [ ]:
from google.colab import files
import os, shutil
ROOT = "/content/talking-avatar"
os.makedirs(f"{ROOT}/photos", exist_ok=True); os.makedirs(f"{ROOT}/voice_samples", exist_ok=True)

print("Upload the PHOTO:")
name, _ = next(iter(files.upload().items()))
PHOTO = f"{ROOT}/photos/photo{os.path.splitext(name)[1].lower()}"
shutil.move(name, PHOTO)

print("Upload the VOICE sample:")
name, _ = next(iter(files.upload().items()))
VOICE = f"{ROOT}/voice_samples/voice{os.path.splitext(name)[1].lower()}"
shutil.move(name, VOICE)
print(PHOTO, VOICE)

### (Optional) Save results to Google Drive so a disconnect doesn't lose them

In [ ]:
OUT_DIR = "/content/talking-avatar/output"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/talking-avatar-output"
except Exception as e:
    print("Drive not mounted, using local disk:", e)
os.makedirs(OUT_DIR, exist_ok=True)
print("results go to", OUT_DIR)

## 7. Test: one short chunk
About 3 seconds of video. **Note the time it takes** (printed at the end) and view the result before rendering more.
`--infinitetalk-mode clip` = a single chunk.

In [ ]:
import subprocess, time, os
PY = "/content/talking-avatar/.venv/bin/python"
SCENE = ("A person speaking warmly and confidently to the camera, natural facial expressions, "
         "subtle head movements, direct eye contact.")   # see the note under section 8


def render(text, out, mode="streaming"):
    cmd = [PY, "make_avatar.py", "--engine", "infinitetalk", "--infinitetalk-accel", "lightx2v",
           "--infinitetalk-mode", mode, "--photo", PHOTO, "--voice", VOICE, "--text", text,
           "--lang", "en", "--scene-prompt", SCENE, "--out", out]
    t = time.time()
    r = subprocess.run(cmd, cwd="/content/talking-avatar", env={**os.environ, "PYTHONUNBUFFERED": "1"})
    print(f"exit {r.returncode} after {(time.time() - t) / 60:.1f} min -> {out}")
    return r.returncode == 0


ok = render("Thank you for being here.", f"{OUT_DIR}/test.mp4", mode="clip")
if ok:
    from IPython.display import Video
    display(Video(f"{OUT_DIR}/test.mp4", embed=True, width=400))

## 8. Render your speech
Edit `SECTIONS` - one string per section. Keep each section to **30-60 seconds of speech** (about 80-150 words):
InfiniteTalk drifts in colour beyond ~1 minute from a single photo, so each section restarts from your photo.
Finished sections are skipped, so you can re-run this cell after a disconnect.

Time estimate: total minutes ~= (chunks) x (minutes per chunk from the test), where a chunk is ~3 s of speech.

The scene prompt (`SCENE` above) shapes expression and head motion; the background comes from your photo, and
the fast mode follows the prompt only loosely.

In [ ]:
SECTIONS = [
    "So let me finish where I started. Thank you. Let's go build something that lasts. Thank you.",
]

import os
done = []
for i, text in enumerate(SECTIONS, 1):
    out = f"{OUT_DIR}/part_{i:02d}.mp4"
    if os.path.exists(out):
        print("skip", out); done.append(out); continue
    print(f"=== section {i}/{len(SECTIONS)}")
    if not render(text, out):
        print("stopped - fix the error above and re-run this cell"); break
    done.append(out)

## 9. Join the sections and download

In [ ]:
import subprocess, os
from google.colab import files
parts = [f"{OUT_DIR}/part_{i:02d}.mp4" for i in range(1, len(SECTIONS) + 1)
         if os.path.exists(f"{OUT_DIR}/part_{i:02d}.mp4")]
final = f"{OUT_DIR}/final.mp4"
if len(parts) == 1:
    final = parts[0]
else:
    listing = f"{OUT_DIR}/parts.txt"
    open(listing, "w").write("".join(f"file '{p}'\n" for p in parts))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", listing,
                    "-c", "copy", final], check=True)
print(final)
files.download(final)